# **Preprocesamiento**

---



Importamos librerías

In [ ]:
import re
import string
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

Cargamos el Dataset

In [ ]:
df = pd.read_csv("dataset_transacciones_validado.csv")

Verficamos estructura del dataset

In [ ]:
df.head()

,descripcion,categoria
0,SAGA FALABELLA AUT641073,Compras
1,DEB BATA,Compras
2,COMPRA JOCKEY PLAZA,Compras
3,TRANSF A TAXI DATUM,Transporte
4,IBERDROLA SEP,Servicios


Definir función de limpieza

In [ ]:
def limpiar_texto(texto):
    texto = str(texto).lower()

    # eliminar espacios repetidos
    texto = re.sub(r"\s+", " ", texto)

    # eliminar signos de puntuación
    texto = texto.translate(str.maketrans("", "", string.punctuation))

    return texto.strip()

Aplicamos la limpieza

In [ ]:
df["descripcion_limpia"] = df["descripcion"].apply(limpiar_texto)

Verificamos

In [ ]:
df.head()

,descripcion,categoria,descripcion_limpia
0,SAGA FALABELLA AUT641073,Compras,saga falabella aut641073
1,DEB BATA,Compras,deb bata
2,COMPRA JOCKEY PLAZA,Compras,compra jockey plaza
3,TRANSF A TAXI DATUM,Transporte,transf a taxi datum
4,IBERDROLA SEP,Servicios,iberdrola sep


Revisar que no existan textos vacíos

In [ ]:
(df["descripcion_limpia"] == "").sum()

np.int64(0)

**Codificamos las categorias**

*Las librerías de Scikit-Learn trabajan con números*

In [ ]:
encoder = LabelEncoder()

df["categoria_id"] = encoder.fit_transform(df["categoria"])

In [ ]:
df.head()

,descripcion,categoria,descripcion_limpia,categoria_id
0,SAGA FALABELLA AUT641073,Compras,saga falabella aut641073,1
1,DEB BATA,Compras,deb bata,1
2,COMPRA JOCKEY PLAZA,Compras,compra jockey plaza,1
3,TRANSF A TAXI DATUM,Transporte,transf a taxi datum,7
4,IBERDROLA SEP,Servicios,iberdrola sep,6


Guardamos la correspondencia de categorías

*Será importante para cuando el modelo haga predicciones*

In [ ]:
categorias = pd.DataFrame({
    "Categoria": encoder.classes_,
    "ID": range(len(encoder.classes_))
})

categorias

,Categoria,ID
0,Alimentación,0
1,Compras,1
2,Educación,2
3,Ocio,3
4,Otros,4
5,Salud,5
6,Servicios,6
7,Transporte,7
8,Vivienda,8


Guardamos como diccionario

In [ ]:
id_categoria = dict(zip(
    encoder.transform(encoder.classes_),
    encoder.classes_
))

id_categoria

{np.int64(0): 'Alimentación',
 np.int64(1): 'Compras',
 np.int64(2): 'Educación',
 np.int64(3): 'Ocio',
 np.int64(4): 'Otros',
 np.int64(5): 'Salud',
 np.int64(6): 'Servicios',
 np.int64(7): 'Transporte',
 np.int64(8): 'Vivienda'}

Separamos Variables

In [ ]:
X = df["descripcion_limpia"]

y = df["categoria_id"]

Dividimos entrenamiento y prueba

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

Verificamos tamaños

In [ ]:
print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)

Entrenamiento: (6400,)
Prueba: (1600,)


Verificamos balance

In [ ]:
#Entrenamiento
pd.Series(y_train).value_counts(normalize=True)

,proportion
categoria_id,
0,0.2000
1,0.1750
7,0.1500
6,0.1500
3,0.1125
5,0.0875
8,0.0625
2,0.0375
4,0.0250


In [ ]:
#Prueba
pd.Series(y_test).value_counts(normalize=True)

,proportion
categoria_id,
0,0.2000
1,0.1750
7,0.1500
6,0.1500
3,0.1125
5,0.0875
8,0.0625
2,0.0375
4,0.0250


Guardamos los datos preparados

In [ ]:
X_train.to_csv("X_train.csv", index=False)

X_test.to_csv("X_test.csv", index=False)

pd.DataFrame(y_train).to_csv("y_train.csv", index=False)

pd.DataFrame(y_test).to_csv("y_test.csv", index=False)

Guardamos LabelEncoder

In [ ]:
import joblib

joblib.dump(encoder, "label_encoder.pkl")

['label_encoder.pkl']

## **TF-IDF (Term Frequency - Inverse Document Frequency).**

Importamos el vectorizador

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

Creamos el vectorizador

In [ ]:
vectorizer = TfidfVectorizer(
    lowercase=False,
    ngram_range=(1,2),
    min_df=2
)

Ajustar TF-IDF

In [ ]:
X_train_tfidf = vectorizer.fit_transform(X_train)

Transformar el conjunto de prueba

In [ ]:
X_test_tfidf = vectorizer.transform(X_test)

Revisar Dimensiones

In [ ]:
print("Entrenamiento:", X_train_tfidf.shape)
print("Prueba:", X_test_tfidf.shape)

Entrenamiento: (6400, 1243)
Prueba: (1600, 1243)


*Se traduce a:*

* 6400 transacciones para entrenar.

* 1600 para evaluar.

* 1243 términos distintos en el vocabulario.

Ver el tamaño del vocabulario

In [ ]:
len(vectorizer.vocabulary_)

1243

Guardamos el vectorizador

In [ ]:
import joblib

joblib.dump(vectorizer, "tfidf_vectorizer.pkl")

['tfidf_vectorizer.pkl']

Guardamos matrices vectorizadas

In [ ]:
from scipy import sparse

sparse.save_npz("X_train_tfidf.npz", X_train_tfidf)
sparse.save_npz("X_test_tfidf.npz", X_test_tfidf)

In [ ]:
pd.Series(y_train).to_csv("y_train.csv", index=False)
pd.Series(y_test).to_csv("y_test.csv", index=False)

Guardar X_Train_texto para Notebook 04

In [ ]:
X_train.to_csv("X_train_texto.csv", index=False)
X_test.to_csv("X_test_texto.csv", index=False)